# SAGE-Lite B2: Phase 6E — Diagnostic A Probe: PointRend Adaptive Point Refinement Head
**Architecture:** Canonical D4 K2 (ConvNeXtV2-Femto + ViT-Tiny 4 layers, P3-C ASDW-OFF, top_k = 2, router_hidden_dim = 64) + PointRend Head
**Branch:** `crack500-audit` | **Hardware Target:** Google Colab Tesla T4 (16GB VRAM)

### Core Hypothesis (Diagnostic A):
- Uncertainty-driven point sampling (2,048 points: 75% boundary uncertainty + 25% uniform coverage) + 3-layer MLP refinement directly at decoder output resolves edge blurriness and prunes false bridges without causing thin crack break events.
- **Lineage Provenance:** Branches strictly from canonical **Candidate B Stage 1 Checkpoint** (`P3_C_D4_K2_H64_Phase5_SAGELR2e-4_best_model_b2_stage1.pth`).
- **Stage 2 Protocol:** 18 epochs, BS 14, seed 42, AdamW, cosine annealing, decoupled loss (CrackBinaryLoss + Point BCE).

## 1. Environment Setup & Repository Synchronization

In [ ]:
# Cell 1: Hardware & CUDA Verification
!nvidia-smi

In [ ]:
# Cell 2: Google Drive Mount & Repository Synchronization
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if not os.path.exists('sage_lite'):
    !git clone -b crack500-audit https://github.com/bach0823/SAGE_LITE.git sage_lite
else:
    %cd /content/sage_lite
    !git fetch origin crack500-audit
    !git checkout crack500-audit
    !git pull origin crack500-audit

%cd /content/sage_lite
!pip install -q -e .
!pip install -q albumentations timm scipy scikit-learn seaborn matplotlib pandas networkx

In [ ]:
# Cell 3: Dataset Invariant Verification
import os

dataset_root = '/content/dataset/Crack500'
if not os.path.exists(dataset_root):
    print(f'[Dataset] {dataset_root} not found! Extracting/preparing Crack500...')
    !python prepare_data/prepare_crack500.py
else:
    print(f'[Dataset] Found canonical dataset at {dataset_root}')

assert os.path.exists(os.path.join(dataset_root, 'train', 'images')), 'Train images missing!'
assert os.path.exists(os.path.join(dataset_root, 'val', 'images')), 'Val images missing!'
assert os.path.exists(os.path.join(dataset_root, 'test', 'images')), 'Test images missing!'
print('[Dataset] Verification PASS: Train, Val, and Test splits confirmed ready.')

## 2. Checkpoint Provenance & Preflight Verification

In [ ]:
# Cell 4: Verify & Stage Candidate B Stage 1 Checkpoint
import os
import shutil

# Check possible locations for Candidate B Stage 1 checkpoint
candidate_paths = [
    '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4/best_model_b2_stage1.pth',
    '/content/drive/MyDrive/crack_seg/Phase5_Candidate_B/best_model_b2_stage1.pth',
    '/content/checkpoints/P3_C_D4_K2_H64_Phase5_SAGELR2e-4_best_model_b2_stage1.pth',
    '/content/sage_lite/results/checkpoints/P3_C_D4_K2_H64_Phase5_SAGELR2e-4_best_model_b2_stage1.pth'
]

stage1_ckpt_path = None
for p in candidate_paths:
    if os.path.exists(p):
        stage1_ckpt_path = p
        break

target_dir = '/content/sage_lite/results/checkpoints'
os.makedirs(target_dir, exist_ok=True)
canonical_target = os.path.join(target_dir, 'P3_C_D4_K2_H64_Phase5_SAGELR2e-4_best_model_b2_stage1.pth')

if stage1_ckpt_path and stage1_ckpt_path != canonical_target:
    shutil.copyfile(stage1_ckpt_path, canonical_target)
    print(f'[Checkpoint] Staged Candidate B Stage 1 checkpoint from {stage1_ckpt_path} -> {canonical_target}')
elif os.path.exists(canonical_target):
    print(f'[Checkpoint] Canonical Stage 1 checkpoint ready at {canonical_target}')
else:
    raise FileNotFoundError('Candidate B Stage 1 checkpoint not found in Drive or local workspace! Please upload it.')

In [ ]:
# Cell 5: Fail-Fast Preflight Verification (PointRend Verification Suite)
%cd /content/sage_lite
!python scripts/tests/test_phase6e_pointrend_verification.py

## 3. Phase 6E Stage 2 Training (PointRend Refinement Head)

In [ ]:
# Cell 6: Train Phase 6E Stage 2 (PointRend Refinement Head)
%cd /content/sage_lite

output_dir = '/content/drive/MyDrive/crack_seg/Phase6E_PointRend'
stage1_ckpt = '/content/sage_lite/results/checkpoints/P3_C_D4_K2_H64_Phase5_SAGELR2e-4_best_model_b2_stage1.pth'

!python scripts/train_crack.py \
    --config configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase6e_pointrend.yaml \
    --two-stage \
    --stage2-only \
    --checkpoint {stage1_ckpt} \
    --output-dir {output_dir}

## 4. Evaluation & Diagnostic Diagnostics Suite

In [ ]:
# Cell 7: Official Canonical Evaluation (Validation Split & Test Split)
%cd /content/sage_lite
import os

phase6e_dir = '/content/drive/MyDrive/crack_seg/Phase6E_PointRend'
phase6e_ckpt = os.path.join(phase6e_dir, 'best_model_b2_global.pth')
phase6e_cfg = 'configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase6e_pointrend.yaml'

print('[Evaluation] 1/2: Evaluating on Official Validation Set...')
!python scripts/evaluate_crack_official.py \
    --config {phase6e_cfg} \
    --checkpoint {phase6e_ckpt} \
    --split val \
    --output-dir {phase6e_dir}/eval_val

print('\n[Evaluation] 2/2: Evaluating on Official Test Set...')
!python scripts/evaluate_crack_official.py \
    --config {phase6e_cfg} \
    --checkpoint {phase6e_ckpt} \
    --split test \
    --output-dir {phase6e_dir}/eval_test

In [ ]:
# Cell 8: SAGE Routing Diagnostics
%cd /content/sage_lite
import os

phase6e_dir = '/content/drive/MyDrive/crack_seg/Phase6E_PointRend'
phase6e_ckpt = os.path.join(phase6e_dir, 'best_model_b2_global.pth')
phase6e_diag_val = os.path.join(phase6e_dir, 'diagnostics', 'full_val')
phase6e_cfg = 'configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase6e_pointrend.yaml'

print('[Routing Diagnostics] Running analyze_routing.py...')
!python tools/analyze_routing.py \
    --config {phase6e_cfg} \
    --checkpoint {phase6e_ckpt} \
    --output_dir {phase6e_diag_val} \
    --split val

In [ ]:
# Cell 9: Error Analysis & Morphology (False Bridges & Break Events)
%cd /content/sage_lite
import os

phase6e_dir = '/content/drive/MyDrive/crack_seg/Phase6E_PointRend'
phase6e_ckpt = os.path.join(phase6e_dir, 'best_model_b2_global.pth')
phase6e_diag_val = os.path.join(phase6e_dir, 'diagnostics', 'full_val')
phase6e_diag_err = os.path.join(phase6e_dir, 'diagnostics', 'error_analysis')
phase6e_cfg = 'configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase6e_pointrend.yaml'

print('[Error Analysis] Running run_p3_c_error_analysis.py...')
!python tools/run_p3_c_error_analysis.py \
    --config {phase6e_cfg} \
    --checkpoint {phase6e_ckpt} \
    --routing-json {phase6e_diag_val}/routing_statistics.json \
    --output-dir {phase6e_diag_err} \
    --data-root /content/dataset/Crack500

In [ ]:
# Cell 10: Topology Diagnostic (Topological Continuity & Graph Metric)
%cd /content/sage_lite
import os

phase6e_dir = '/content/drive/MyDrive/crack_seg/Phase6E_PointRend'
phase6e_ckpt = os.path.join(phase6e_dir, 'best_model_b2_global.pth')
phase6e_diag_topo = os.path.join(phase6e_dir, 'diagnostics', 'topology')
phase6e_cfg = 'configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase6e_pointrend.yaml'

print('[Topology Diagnostic] Running run_phase6_c_topology_diagnostic.py...')
!python tools/run_phase6_c_topology_diagnostic.py \
    --config {phase6e_cfg} \
    --checkpoint {phase6e_ckpt} \
    --output-dir {phase6e_diag_topo} \
    --split val \
    --data-root /content/dataset/Crack500

## 5. Artifact Archiving & Comparison

In [ ]:
# Cell 11: Zip & Export Full Artifacts for Local Inspection
%cd /content/drive/MyDrive/crack_seg
!zip -r Phase6E_PointRend_Full.zip Phase6E_PointRend
print('\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/Phase6E_PointRend_Full.zip')

# Optional: Tự động tải về máy nếu cần
# from google.colab import files
# files.download('/content/drive/MyDrive/crack_seg/Phase6E_PointRend_Full.zip')

In [ ]:
# Cell 12: Summary Comparison against Candidate B & Phase 6D (CGSR)
import os
import json
import torch
import pandas as pd

experiments = {
    'Candidate B (Phase 5 Baseline)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4/diagnostics/error_analysis/error_analysis_summary.json',
    },
    'Phase 6D (CGSR Skip Refinement)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/Phase6D_CGSR/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/Phase6D_CGSR/diagnostics/error_analysis/error_analysis_summary.json',
    },
    'Phase 6E (PointRend Refinement Head)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/Phase6E_PointRend/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/Phase6E_PointRend/diagnostics/error_analysis/error_analysis_summary.json',
    }
}

summary = []
for name, data in experiments.items():
    row = {'Experiment': name, 'Best Val Dice': None, 'Best Val Loss': None, 'Val IoU': None, 'Status': 'Missing'}
    if os.path.exists(data['ckpt']):
        c = torch.load(data['ckpt'], map_location='cpu', weights_only=False)
        row['Best Val Dice'] = round(float(c.get('best_dice', 0.0)), 4)
        row['Best Val Loss'] = round(float(c.get('best_loss', 0.0)), 4)
        row['Status'] = f"Epoch {c.get('epoch', '?')} (Stage {c.get('stage', '?')})"
    if os.path.exists(data['err_json']):
        with open(data['err_json']) as f:
            ej = json.load(f)
            row['Val IoU'] = round(float(ej.get('overall_metrics', {}).get('mean_iou', 0.0)), 4)
    summary.append(row)

df = pd.DataFrame(summary)
print('\n' + '='*85)
print('PHASE 6E vs BASELINE COMPARATIVE SUMMARY (Crack500 Val 348 Samples)')
print('='*85)
print(df.to_string(index=False))
print('='*85 + '\n')